In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt

# 1. Data Exploration

In [5]:
data = pd.read_csv("../data/PS_20174392719_1491204439457_log.csv")

In [6]:
data.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [7]:
data.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [8]:
data.shape

(6362620, 11)

In [9]:
data.isna().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [12]:
data["isFraud"].value_counts(normalize = True)

isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64


## 1.1 Class Distribution
Fraud accounts for only **0.129%** of transactions. This has a direct consequence: **accuracy is not a valid metric for this problem.** A model that predicts "not fraud" for every single transaction would score 99.87% accuracy while catching zero actual fraud cases — the metric would reward doing nothing useful.
**Implication for evaluation:** this project uses Precision, Recall, F1, and PR-AUC instead of accuracy, since these metrics actually reflect whether fraud is being detected.


In [19]:
data[data["isFraud"]==1]["type"].value_counts()

type
CASH_OUT    4116
TRANSFER    4097
Name: count, dtype: int64

## 1.2 Fraud by Transaction Type and Amount

**Transaction type:**

In [21]:
data.groupby('isFraud')['amount'].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,1.781970e+05,5.962370e+05,0.01,13368.395,74684.72,208364.76,92445516.64
1,8213.0,1.467967e+06,2.404253e+06,0.00,127091.330,441423.44,1517771.48,10000000.00


In [24]:
total_volume = data['amount'].sum()
fraud_volume = data[data['isFraud']==1]['amount'].sum()
print(f"Total volume: ${total_volume:,.2f}")
print(f"Fraud volume: ${fraud_volume:,.2f}")
print(f"Fraud % of total volume: {fraud_volume/total_volume*100:.2f}%")

Total volume: $1,144,392,944,759.77
Fraud volume: $12,056,415,427.84
Fraud % of total volume: 1.05%


Fraud occurs **exclusively** in `TRANSFER` and `CASH_OUT` transactions — zero fraud cases in `PAYMENT`, `CASH_IN`, or `DEBIT`. This matches the known real-world fraud pattern for mobile money: funds are moved to another account (`TRANSFER`), then withdrawn as cash (`CASH_OUT`) before detection. `type` is therefore a very strong feature, and the other three types could in principle be excluded from scoring entirely since they carry zero fraud risk in this data.

**Transaction amount:**

| | count | mean | median (50%) | max |
|---|---|---|---|---|
| Not Fraud (0) | 6,354,407 | $178,197 | $74,685 | $92,445,517 |
| Fraud (1) | 8,213 | $1,467,967 | $441,423 | $10,000,000 |

Fraudulent transactions are **~6-8x larger** than legitimate ones on average, and the gap holds at the median too — not just a few outliers skewing the mean. `amount` is likely to be a strong predictive feature alongside `type`.

*Note: fraud's max amount is suspiciously capped at exactly $10,000,000 — likely an artifact of how the PaySim simulator generated synthetic fraud cases, not a pattern expected in real-world data.*

## 1.3 Financial Impact of Fraud

| Metric | Value |
|---|---|
| Total transaction volume | $1,144,392,944,759.77 |
| Total fraud volume | $12,056,415,427.84 |
| Fraud as % of total volume | **1.05%** |
| Fraud as % of transaction count | 0.13% |

Although fraudulent transactions make up only **0.13% of all transactions by count**, they account for **1.05% of total transaction value** — roughly **8x disproportionate impact**. This confirms the earlier finding: fraud transactions are systematically larger than normal ones, so even though they're rare in number, their financial footprint is meaningfully outsized relative to their frequency.

**Business framing:** detecting even a fraction of these cases has outsized value — missing a single large fraudulent transaction costs far more than missing a typical legitimate one.